In [0]:
from pyspark.sql.functions import col

orders = spark.table("retail_project.bronze.orders") \
    .dropDuplicates(["order_id"]) \
    .withColumn("order_purchase_timestamp", col("order_purchase_timestamp").cast("timestamp")) \
    .withColumn("order_delivered_customer_date", col("order_delivered_customer_date").cast("timestamp")) \
    .filter(col("order_status").isNotNull())

order_items = spark.table("retail_project.bronze.order_items") \
    .dropDuplicates(["order_id", "order_item_id"]) \
    .withColumn("price", col("price").cast("double")) \
    .withColumn("freight_value", col("freight_value").cast("double"))

products = spark.table("retail_project.bronze.products") \
    .dropDuplicates(["product_id"])

category_translation = spark.table("retail_project.bronze.category_translation")

customers = spark.table("retail_project.bronze.customers") \
    .dropDuplicates(["customer_id"])

silver_fact = (
    order_items
    .join(orders, "order_id", "inner")
    .join(products, "product_id", "left")
    .join(category_translation, "product_category_name", "left")
    .join(customers, "customer_id", "left")
    .select(
        "order_id", "order_item_id", "product_id",
        "product_category_name_english",
        "customer_id", "customer_state",
        "order_purchase_timestamp", "order_delivered_customer_date",
        "price", "freight_value", "order_status"
    )
    .filter(col("order_status") == "delivered")
)

silver_fact.write.format("delta").mode("overwrite") \
    .saveAsTable("retail_project.silver.order_items_enriched")

print(f"Silver fact table rows: {silver_fact.count()}")

In [0]:
from pyspark.sql.functions import count, sum as spark_sum

null_check = silver_fact.select(
    count("*").alias("total_rows"),
    spark_sum(col("price").isNull().cast("int")).alias("null_prices"),
    spark_sum(col("product_category_name_english").isNull().cast("int")).alias("null_categories")
)
null_check.show()